# Normilizer for Woolf's The Boyage Out as baseline

### [—————————————pipeline—————————————]
### »——raw—»—clean—»—NORMALIZE—»—dataframe——»–eda–»

Solid and clean copy. Here are some characteristics. 

## Before
- Cleanish copy
- kept line breaks
- no marginalia or paratext, except: 
    - Book breakers

## After
- clean text: only alphabetic characters
- kept syntactical punctuation (. , ; : - " ') for future analysis
- kept one line = one verse; a line break structure for prosodic analysis
- kept book titles (because some book's first lines are repeated 19 times in the text)
- removed macron and accents
- removed line numbers
- removed numeral digits
- strpped empty lines


In [14]:
translator = "Woolf"
filepath = f"/Users/debr/odysseys_en/cleaned_txts/Odyssey_{translator}_Cleaned.txt"

In [15]:
# Step 1: read txt and remove book's headers
with open(filepath, 'r') as file:
    lines = file.readlines()

new_lines = [line for line in lines if line.strip() != ''] # Remove empty lines

# Print a preview of the processed text
text = "".join(new_lines)
print(text[:100])

CHAPTER I
As the streets that lead from the Strand to the Embankment are very
narrow, it is better n


## Path A: Controlled normalization

A transparent, "manual," normalization using regex.  

#### Output file: "Odyssey_{--translator--}_Normalized_v1.txt"

In [16]:
# Step 2: Normalize the text
import re

# Removing numeral digits
def remove_digits(text):
    """
    Remove all digits from a string.
    """
    if isinstance(text, list):
        # If text is a list, apply the function to each element
        return [remove_digits(line) for line in text]
    else:
        # Remove digits using regex
        return re.sub(r'\d', '', text)

# Removing diacritics, macrons, circumflex, dieresis/umlaut, etc
def normalize_text(text):
    """
    Normalize text by replacing diacritics with their base characters.
    """
    # Dictionary mapping diacritics to their base characters
    replacements = {
        # Dieresis/umlaut characters
        'ä': 'a', 'ë': 'e', 'ï': 'i', 'ö': 'o', 'ü': 'u',
        'Ä': 'A', 'Ë': 'E', 'Ï': 'I', 'Ö': 'O', 'Ü': 'U',
        
        # Circumflex characters
        'â': 'a', 'ê': 'e', 'î': 'i', 'ô': 'o', 'û': 'u',
        'Â': 'A', 'Ê': 'E', 'Î': 'I', 'Ô': 'O', 'Û': 'U',
        
        # Macron characters
        'ā': 'a', 'ē': 'e', 'ī': 'i', 'ō': 'o', 'ū': 'u',
        'Ā': 'A', 'Ē': 'E', 'Ī': 'I', 'Ō': 'O', 'Ū': 'U',
        
        # Even more diacritics
        # Acute
        'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u',
        # Grave
        'à': 'a', 'è': 'e', 'ì': 'i', 'ò': 'o', 'ù': 'u',
    }
    
    # Replacement time!
    for original, replacement in replacements.items():
        text = text.replace(original, replacement)
    
    return text
    

In [17]:
print(text[:1000])

CHAPTER I
As the streets that lead from the Strand to the Embankment are very
narrow, it is better not to walk down them arm-in-arm. If you persist,
lawyers’ clerks will have to make flying leaps into the mud; young lady
typists will have to fidget behind you. In the streets of London where
beauty goes unregarded, eccentricity must pay the penalty, and it is
better not to be very tall, to wear a long blue cloak, or to beat the
air with your left hand.
One afternoon in the beginning of October when the traffic was becoming
brisk a tall man strode along the edge of the pavement with a lady on
his arm. Angry glances struck upon their backs. The small, agitated
figures—for in comparison with this couple most people looked
small—decorated with fountain pens, and burdened with despatch-boxes,
had appointments to keep, and drew a weekly salary, so that there was
some reason for the unfriendly stare which was bestowed upon Mr.
Ambrose’s height and upon Mrs. Ambrose’s cloak. But some enchantmen

In [18]:
# Apply the normalization to the text
text = remove_digits(text)
text = normalize_text(text)

In [19]:
import re

def clean_text(text):
    """
    Removes text inside brackets, extra spaces, and spaces after newlines.

    Args:
        text (str): The input text to clean.

    Returns:
        str: The cleaned text.
    """
    if not isinstance(text, str):
        return text  # Return as is if not a string

    # Remove text inside brackets (multi-line supported)
    text = re.sub(r'\[.*?\]', '', text, flags=re.DOTALL)

    # Remove spaces after newlines
    text = re.sub(r'\n\s+', '\n', text)

    return text.strip()

cleaned_text = clean_text(text)
print(cleaned_text)

CHAPTER I
As the streets that lead from the Strand to the Embankment are very
narrow, it is better not to walk down them arm-in-arm. If you persist,
lawyers’ clerks will have to make flying leaps into the mud; young lady
typists will have to fidget behind you. In the streets of London where
beauty goes unregarded, eccentricity must pay the penalty, and it is
better not to be very tall, to wear a long blue cloak, or to beat the
air with your left hand.
One afternoon in the beginning of October when the traffic was becoming
brisk a tall man strode along the edge of the pavement with a lady on
his arm. Angry glances struck upon their backs. The small, agitated
figures—for in comparison with this couple most people looked
small—decorated with fountain pens, and burdened with despatch-boxes,
had appointments to keep, and drew a weekly salary, so that there was
some reason for the unfriendly stare which was bestowed upon Mr.
Ambrose’s height and upon Mrs. Ambrose’s cloak. But some enchantmen

In [20]:
# Create output directory if it doesn't exist
import os
output_filepath = f"/Users/debr/odysseys_en/normalized_txts/Odyssey_{translator}_Normalized_v1.txt"
os.makedirs(os.path.dirname(output_filepath), exist_ok=True)

# Write to a new file 
with open(output_filepath, 'w') as file:
    file.writelines(cleaned_text)

## Path B: Broad-brush method normalization

An indiscriminate approach to normalization using the unicodedata Python module.
Fast an efficient but we don't control all the changes as method A.

#### Output file: "Odyssey_Green_Normalized_v2.txt"

In [21]:
import os
import unicodedata
import re

def normalize_text_unicodedata(text):
    """
    Normalize text using Python's unicodedata module.
    This handles all diacritics in a standardized way.
    """
    # NFD decomposes characters into base characters and combining marks, ie, "é" -> "e" + "◌́" 
    # Then we filter out the combining marks (category starts with 'M')
    return ''.join(c for c in unicodedata.normalize('NFD', text)
                  if not unicodedata.category(c).startswith('M'))

def remove_digits(text):
    """
    Remove all digits from a string.
    """
    if isinstance(text, list):
        # If text is a list, apply the function to each element
        return [remove_digits(line) for line in text]
    else:
        # Remove digits using regex
        return re.sub(r'\d', '', text)

# Read clean text again
filepath = f"/Users/debr/odysseys_en/cleaned_txts/Odyssey_{translator}_Cleaned.txt"

with open(filepath, 'r', encoding='utf-8') as file:
    lines = file.readlines()

# Filter out the book headers
#book_headers = [f'Book {i}' for i in range(1, 25)]
#filtered_lines = [line for line in lines if line.strip() not in book_headers]
filtered_lines = [line for line in lines if line.strip() != '']

# Normalize each line
normalized_lines = [normalize_text_unicodedata(line) for line in filtered_lines]

# Remove digits
final_lines = remove_digits(normalized_lines)

text_uni = "".join(final_lines)

In [22]:
# Replace consecutive empty lines with a single newline
text_uni = re.sub(r'\n+', '\n', text_uni)
print(text_uni[:1000])

CHAPTER I
As the streets that lead from the Strand to the Embankment are very
narrow, it is better not to walk down them arm-in-arm. If you persist,
lawyers’ clerks will have to make flying leaps into the mud; young lady
typists will have to fidget behind you. In the streets of London where
beauty goes unregarded, eccentricity must pay the penalty, and it is
better not to be very tall, to wear a long blue cloak, or to beat the
air with your left hand.
One afternoon in the beginning of October when the traffic was becoming
brisk a tall man strode along the edge of the pavement with a lady on
his arm. Angry glances struck upon their backs. The small, agitated
figures—for in comparison with this couple most people looked
small—decorated with fountain pens, and burdened with despatch-boxes,
had appointments to keep, and drew a weekly salary, so that there was
some reason for the unfriendly stare which was bestowed upon Mr.
Ambrose’s height and upon Mrs. Ambrose’s cloak. But some enchantmen

In [23]:
# Create output directory if it doesn't exist
output_filepath = f"/Users/debr/odysseys_en/normalized_txts/Odyssey_{translator}_Normalized_v2.txt"
os.makedirs(os.path.dirname(output_filepath), exist_ok=True)

# Write to a new file
with open(output_filepath, 'w', encoding='utf-8') as file:
    file.writelines(text_uni)

print(f"Normalization complete. File saved to: {output_filepath}")

Normalization complete. File saved to: /Users/debr/odysseys_en/normalized_txts/Odyssey_Woolf_Normalized_v2.txt


In [24]:
print("length after controlled normalization:", len(text))
print("Length after cleaned text normalization:", len(cleaned_text))
print("Length after broad-brush normalization:", len(text_uni))

length after controlled normalization: 760920
Length after cleaned text normalization: 760872
Length after broad-brush normalization: 760920


In [25]:
# count new lines "\n"
def count_line_breaks(text):
    line_breaks = text.count('\n')
    return line_breaks

# Usage example
lines_my_norm = count_line_breaks(text)
lines_cleaned = count_line_breaks(cleaned_text)
lines_unicode = count_line_breaks(text_uni)
print(f"Number of lines in my custom normalization: {lines_my_norm}")
print(f"Number of lines in cleaned text: {lines_cleaned}")
print(f"Number of lines with unicode normalization: {lines_my_norm}")

Number of lines in my custom normalization: 12176
Number of lines in cleaned text: 12176
Number of lines with unicode normalization: 12176
